# 파이선 기본으로 연결하기

In [59]:
# uv add oracledb

# 1. DB 연결

In [60]:
import oracledb

conn = oracledb.connect(
    user="joy",
    password="Joy1234",
    dsn="localhost:1521/movie_db"
)

print("Oracle DB 연결 성공")

cursor = conn.cursor()

print("Oracle Cursor 생성 성공")


Oracle DB 연결 성공
Oracle Cursor 생성 성공


In [61]:
# MEMBER 테이블 존재 여부 확인
cursor.execute("""
    SELECT COUNT(*)
    FROM USER_TABLES
    WHERE TABLE_NAME = 'MEMBER'
""")

<oracledb.Cursor on <oracledb.Connection to joy@localhost:1521/movie_db>>

In [62]:
exists = cursor.fetchone()[0]
exists

1

In [63]:
def create_member_table_if_not_exists(cursor):
    cursor.execute("""
        SELECT COUNT(*)
        FROM USER_TABLES
        WHERE TABLE_NAME = 'MEMBER'
    """)

    exists = cursor.fetchone()[0]

    if exists > 0:
        print("MEMBER 테이블이 이미 존재합니다.")
        return

    cursor.execute("""
        CREATE TABLE MEMBER (
            MEMBER_ID      NUMBER          CONSTRAINT PK_MEMBER PRIMARY KEY,
            MEMBER_NAME    VARCHAR2(50)    NOT NULL,
            EMAIL          VARCHAR2(100)   CONSTRAINT UQ_MEMBER_EMAIL UNIQUE,
            GRADE          VARCHAR2(10)    DEFAULT 'BASIC',
            JOIN_DATE      DATE            DEFAULT SYSDATE,

            CONSTRAINT CK_MEMBER_GRADE
                CHECK (GRADE IN ('BASIC', 'VIP', 'VVIP'))
        )
        TABLESPACE USERS
    """)

    print("MEMBER 테이블을 생성했습니다.")

In [64]:
create_member_table_if_not_exists(cursor)

MEMBER 테이블이 이미 존재합니다.


## 2. Create (데이터 생성)

In [65]:
members = [
    (1, "홍길동", "hong@example.com", "BASIC"),
    (2, "김철수", "kim@example.com", "VIP"),
    (3, "이영희", "lee@example.com", "VVIP"),
    (4, "박민수", "park@example.com", "BASIC"),
    (5, "최수진", "choi@example.com", "VIP"),
]

sql = """
    INSERT INTO MEMBER (
        MEMBER_ID,
        MEMBER_NAME,
        EMAIL,
        GRADE
    )
    VALUES (
        :1,
        :2,
        :3,
        :4
    )
"""

cursor.executemany(sql, members)

conn.commit()

print(f"{cursor.rowcount}건의 데이터가 입력되었습니다.")

DatabaseError: ORA-01950: The user JOY has insufficient quota on tablespace SYSTEM.
Help: https://docs.oracle.com/error-help/db/ora-01950/

In [ ]:
conn.close()